In [ ]:
# Install required libraries for transformer-based model training and dataset loading
!pip install transformers accelerate datasets

In [ ]:
import sys
import logging
from functools import partial
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset

logger = logging.getLogger(__name__)

<table><tr><td style="background-color:white; padding:10px;">
  <img src="https://lh7-rt.googleusercontent.com/docsz/AD_4nXeMeLo6dJ3-rUblI3auCmzpsjC5Y1FrJAIkZtL0zZ1tXACWzI0lBpZJqZ1DC11m9yP0H2Vx7vz6xRyemQv7qlpDCpB40bwXOEhRwGZgcj0ESTFP23nI22dO4wNCJDBa0KeVOBeFSw?key=_O6ahQ_HWoQw21JjXaX7LnOa" />
</td></tr></table>


# 📘 Introduction to Instruction Tuning

---

Instruction tuning is the process of fine-tuning a pretrained language model on **(instruction, response)** pairs so it learns to **follow natural language requests** rather than just predict the next token.

In this notebook, we instruction-tune **EleutherAI/Pythia-2.8B** on the **Databricks Dolly 15k** dataset.

---

📌 What this notebook covers:

| Topic | Description |
|-------|-------------|
| **Prompt Formatting** | Structured templates with special tokens (`### Instruction:`, `### Response:`, `### End`) |
| **Loss Masking** | Train only on response tokens, not the prompt |
| **Autoregressive Generation** | Inference with the fine-tuned model |

---

💬 Sample Formatted Training Example:
```
Below is an instruction that describes a task. Write a response that appropriately completes the request.
### Instruction:
When did the Apollo 11 mission land on the Moon?
### Response:
The Apollo 11 mission landed on the Moon on July 20, 1969.
### End
```

---

> 🔑 **Pretraining vs Instruction Tuning**
>
> **Pretraining** teaches a model general language by predicting the next token on massive, unstructured text (books, web pages, etc.) — broad knowledge, but no sense of how to follow instructions.
>
> **Instruction Tuning** fine-tunes that pretrained model on structured (instruction, response) pairs, teaching it *how* to behave as a helpful assistant. Techniques like **loss masking** ensure the model only learns to generate the response, not memorize the prompt.

# 1) Training

## 1.1 Initialization

In [ ]:
# --- Special tokens that structure the instruction-tuning prompt format ---
INSTRUCTION_KEY = "### Instruction:"
INPUT_KEY = "### Input:"
RESPONSE_KEY = "### Response:"
END_KEY = "### End"
RESPONSE_KEY_NL = f"{RESPONSE_KEY}\n"

## 1.2 Model Loading

In [ ]:
INPUT_MODEL = "EleutherAI/pythia-2.8b"


def load_tokenizer(pretrained_model_name_or_path):
    """Load tokenizer and register special tokens for the instruction format."""
    tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.add_special_tokens(
        {"additional_special_tokens": [END_KEY, INSTRUCTION_KEY, RESPONSE_KEY_NL]}
    )
    return tokenizer


def load_model(pretrained_model_name_or_path, gradient_checkpointing=False):
    """Load causal LM. Gradient checkpointing trades compute for memory."""
    model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path,
        trust_remote_code=True,
        use_cache=not gradient_checkpointing,
    )
    return model


def get_model_tokenizer(pretrained_model_name_or_path, gradient_checkpointing=False):
    """Load model + tokenizer and resize embeddings for new special tokens."""
    tokenizer = load_tokenizer(pretrained_model_name_or_path)
    model = load_model(
        pretrained_model_name_or_path, gradient_checkpointing=gradient_checkpointing
    )

    # Resize embedding layer to accommodate added special tokens
    # Output: (vocab_size + num_new_tokens, model_dim)
    model.resize_token_embeddings(len(tokenizer))
    return model, tokenizer


# --- Instantiate model and tokenizer ---
model, tokenizer = get_model_tokenizer(
    pretrained_model_name_or_path=INPUT_MODEL, gradient_checkpointing=True
)

# Retrieve max sequence length from model config (used for truncation)
max_length = getattr(model.config, "max_position_embeddings", 2048)

# Optional: torch.compile for faster execution on supported platforms
if torch.__version__ >= "2" and sys.platform != "win32":
    model = torch.compile(model)

## 1.3 Data Processing

In [ ]:
# System instruction prepended to every prompt
INTRO_BLURB = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request."
)

# --- Prompt template WITHOUT additional context ---
PROMPT_NO_INPUT_FORMAT = """{intro}
{instruction_key}
{instruction}
{response_key}
{response}
{end_key}""".format(
    intro=INTRO_BLURB,
    instruction_key=INSTRUCTION_KEY,
    instruction="{instruction}",
    response_key=RESPONSE_KEY,
    response="{response}",
    end_key=END_KEY,
)

# --- Prompt template WITH additional context/input ---
PROMPT_WITH_INPUT_FORMAT = """{intro}
{instruction_key}
{instruction}
{input_key}
{input}
{response_key}
{response}
{end_key}""".format(
    intro=INTRO_BLURB,
    instruction_key=INSTRUCTION_KEY,
    instruction="{instruction}",
    input_key=INPUT_KEY,
    input="{input}",
    response_key=RESPONSE_KEY,
    response="{response}",
    end_key=END_KEY,
)


def load_training_dataset(path_or_dataset="databricks/databricks-dolly-15k"):
    """Load Dolly 15k and format each record into a full prompt string."""
    dataset = load_dataset(path_or_dataset)["train"]

    def _add_text(rec):
        instruction = rec["instruction"]
        response = rec["response"]
        context = rec.get("context")
        if context:
            rec["text"] = PROMPT_WITH_INPUT_FORMAT.format(
                instruction=instruction, response=response, input=context
            )
        else:
            rec["text"] = PROMPT_NO_INPUT_FORMAT.format(
                instruction=instruction, response=response
            )
        return rec

    dataset = dataset.map(_add_text)
    return dataset


def preprocess_batch(batch, tokenizer, max_length):
    """Tokenize a batch of prompt strings, truncating to max_length.

    Input:  batch["text"] — list of strings (batch_num,)
    Output: {"input_ids": (batch_num, seq_len), "attention_mask": (batch_num, seq_len)}
    """
    return tokenizer(batch["text"], max_length=max_length, truncation=True)


def preprocess_dataset(tokenizer, max_length):
    """Full pipeline: load -> tokenize -> filter truncated records -> shuffle."""
    dataset = load_training_dataset()
    _preprocessing_function = partial(
        preprocess_batch, max_length=max_length, tokenizer=tokenizer
    )

    # Tokenize all records, removing raw text columns
    dataset = dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=["instruction", "context", "response", "text", "category"],
    )

    # Remove truncated records — they are missing the END_KEY token
    dataset = dataset.filter(lambda rec: len(rec["input_ids"]) < max_length)
    dataset = dataset.shuffle()
    return dataset


class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    """Custom collator that masks prompt tokens so the loss is computed only on
    the model's response (tokens after '### Response:\\n').

    Labels shape: (batch_num, seq_len_padded)
    Prompt tokens are set to -100 so cross-entropy loss ignores them.
    """

    def torch_call(self, examples):
        # Standard collation: pads and creates labels
        # Input/Output: {"input_ids": (batch_num, seq_len_padded),
        #                "attention_mask": (batch_num, seq_len_padded),
        #                "labels": (batch_num, seq_len_padded)}
        batch = super().torch_call(examples)

        response_token_ids = self.tokenizer.encode(RESPONSE_KEY_NL)
        labels = batch["labels"].clone()

        # Input: labels (batch_num, seq_len_padded)
        # Output: labels (batch_num, seq_len_padded) with prompt masked
        for i in range(len(examples)):
            response_token_ids_start_idx = None

            # Find the first occurrence of the response key token
            for idx in np.where(batch["labels"][i] == response_token_ids[0])[0]:
                response_token_ids_start_idx = idx
                break

            if response_token_ids_start_idx is None:
                raise RuntimeError(
                    f"Could not find response key {response_token_ids} "
                    f"in token IDs {batch['labels'][i]}"
                )

            response_token_ids_end_idx = response_token_ids_start_idx + 1

            # Mask all tokens up to and including the response key with -100
            labels[i, :response_token_ids_end_idx] = -100

        batch["labels"] = labels
        return batch


# --- Run preprocessing and split into train/test ---
processed_dataset = preprocess_dataset(tokenizer=tokenizer, max_length=max_length)
split_dataset = processed_dataset.train_test_split(test_size=1000)

# Inspect a single tokenized sample
for k, v in next(iter(processed_dataset)).items():
    print(f"{k}: {v}\n")

# --- Instantiate the custom data collator (no MLM, pad to multiple of 8) ---
data_collator = DataCollatorForCompletionOnlyLM(
    tokenizer=tokenizer, mlm=False, return_tensors="pt", pad_to_multiple_of=8
)

## 1.4 Trainer

In [ ]:
local_output_dir = "/logs/"

training_args = TrainingArguments(
    output_dir=local_output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    fp16=False,
    bf16=False,
    learning_rate=1e-5,
    num_train_epochs=5,
    gradient_checkpointing=True,
    logging_dir=f"{local_output_dir}/runs",
    logging_strategy="steps",
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=10,
    load_best_model_at_end=False,
    report_to="tensorboard",
    disable_tqdm=True,
    remove_unused_columns=False,
    warmup_steps=0,
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    data_collator=data_collator,
)

# Launch training and save the fine-tuned model
trainer.train()
trainer.save_model(output_dir=local_output_dir)

# 2) Generation

In [ ]:
# Generation-time prompt template (no response body or end token — model generates those)
PROMPT_FOR_GENERATION_FORMAT = """{intro}
{instruction_key}
{instruction}
{response_key}
""".format(
    intro=INTRO_BLURB,
    instruction_key=INSTRUCTION_KEY,
    instruction="{instruction}",
    response_key=RESPONSE_KEY,
)


def preprocess(tokenizer, instruction_text):
    """Tokenize a single instruction into model inputs for generation.

    Output: inputs["input_ids"] -> (1, seq_len)
            inputs["attention_mask"] -> (1, seq_len)
    """
    prompt_text = PROMPT_FOR_GENERATION_FORMAT.format(instruction=instruction_text)
    inputs = tokenizer(prompt_text, return_tensors="pt")
    inputs["prompt_text"] = prompt_text
    inputs["instruction_text"] = instruction_text
    return inputs


def forward(model, tokenizer, model_inputs, max_length=100):
    """Run autoregressive generation given tokenized inputs.

    Input:  input_ids (batch_num, seq_len)
    Output: generated_sequence (batch_num, num_return_sequences, generated_seq_len)
    """
    input_ids = model_inputs["input_ids"]  # (batch_num, seq_len)
    attention_mask = model_inputs.get("attention_mask", None)
    batch_num = input_ids.shape[0]

    # Generate tokens autoregressively
    # Input: (batch_num, seq_len)
    # Output: (batch_num, generated_seq_len)
    generated_sequence = model.generate(
        input_ids=input_ids.to(model.device),
        attention_mask=attention_mask.to(model.device) if attention_mask is not None else None,
        pad_token_id=tokenizer.pad_token_id,
        max_length=max_length,
    )

    # Reshape: (batch_num * num_return_sequences, generated_seq_len)
    #       -> (batch_num, num_return_sequences, generated_seq_len)
    out_b = generated_sequence.shape[0]
    generated_sequence = generated_sequence.reshape(
        batch_num, out_b // batch_num, *generated_sequence.shape[1:]
    )

    instruction_text = model_inputs.get("instruction_text", None)
    return {
        "generated_sequence": generated_sequence,
        "input_ids": input_ids,
        "instruction_text": instruction_text,
    }


# --- Run a sample generation ---
text = "Give me best 30s advice"
pre_process_result = preprocess(tokenizer, text)
print(pre_process_result["input_ids"])  # (1, seq_len)
print(pre_process_result["prompt_text"])
model_result = forward(model, tokenizer, pre_process_result)

In [ ]:
# Extract the response text from generated tokens

def get_special_token_id(tokenizer, key):
    """Get the single token ID for a string registered as a special token."""
    token_ids = tokenizer.encode(key)
    if len(token_ids) > 1:
        raise ValueError(f"Expected a single token for '{key}' but found {token_ids}")
    return token_ids[0]


def postprocess(tokenizer, model_outputs, return_full_text=False):
    """Decode generated sequences by locating the response/end boundaries.

    Extracts text between '### Response:\\n' and '### End' tokens.
    """

    response_key_token_id = get_special_token_id(tokenizer, RESPONSE_KEY_NL)
    end_key_token_id = get_special_token_id(tokenizer, END_KEY)

    # generated_sequence shape: (num_return_sequences, generated_seq_len)
    generated_sequence = model_outputs["generated_sequence"][0]
    instruction_text = model_outputs["instruction_text"]
    generated_sequence = generated_sequence.numpy().tolist()
    records = []

    for sequence in generated_sequence:
        decoded = None

        # Find where "### Response:\n" appears in the generated tokens
        try:
            response_pos = sequence.index(response_key_token_id)
        except ValueError:
            logger.warning(
                f"Could not find response key {response_key_token_id} in: {sequence}"
            )
            response_pos = None

        if response_pos is not None:
            # Find where "### End" appears (may be absent if output was truncated)
            try:
                end_pos = sequence.index(end_key_token_id)
            except ValueError:
                logger.warning("Could not find end key, the output is truncated!")
                end_pos = None

            # Decode only the response tokens between response key and end key
            decoded = tokenizer.decode(sequence[response_pos + 1 : end_pos]).strip()

        if return_full_text:
            decoded = f"{instruction_text}\n{decoded}"

        records.append({"generated_text": decoded})

    return records


# --- Run post-processing and print the final response ---
final_output = postprocess(tokenizer, model_result, return_full_text=False)
print(final_output)